# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

Ranked Actions:

    QUEUE_FOR_EDITORIAL_REFRESH: High confidence of decay; prioritize for immediate content audit.

    MONITOR_NEXT_30_DAYS: Borderline cases; wait for more data to confirm the downward trend.

Reason Codes:

    STALE_HIGH_EXPOSURE: Page is older than 180 days with high historical volume masking gradual decay.

    ENGAGEMENT_FRICTION: CTR dropping significantly ahead of organic rank changes.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatically define the playbook mapping to ensure consistency in downstream tasks
playbook_mapping = {
    "primary_action": "QUEUE_FOR_EDITORIAL_REFRESH",
    "primary_reason": "STALE_HIGH_EXPOSURE",
    "secondary_action": "MONITOR_NEXT_30_DAYS",
    "secondary_reason": "ENGAGEMENT_FRICTION"
}
print("Playbook Actions & Codes Initialized:", playbook_mapping)

Playbook Actions & Codes Initialized: {'primary_action': 'QUEUE_FOR_EDITORIAL_REFRESH', 'primary_reason': 'STALE_HIGH_EXPOSURE', 'secondary_action': 'MONITOR_NEXT_30_DAYS', 'secondary_reason': 'ENGAGEMENT_FRICTION'}


## 2. Intended use and limits

Intended Use: Directional decision-support to help SEO and editorial teams prioritize their manual content audit workflows. It tells them where to look, not what to write.

System Limits: The model strictly observes historical correlation but cannot distinguish between true structural content decay and natural seasonal demand dips (e.g., a "Summer Travel Guide" naturally losing traffic in December).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the operational bounds of the model
model_limits = [
    "Cannot detect seasonality natively.",
    "Cannot identify competitor brand cannibalization.",
    "Assumes tracking data (GA4/GSC) is perfectly stable."
]
print("Operational Limits Registered:")
for limit in model_limits:
    print(f"- {limit}")

Operational Limits Registered:
- Cannot detect seasonality natively.
- Cannot identify competitor brand cannibalization.
- Assumes tracking data (GA4/GSC) is perfectly stable.


## 3. Human review + the no-go list
Human Review Requirements: Editors must independently verify if search intent has shifted (e.g., Google now answers the query directly via an AI Overview) before altering the text.

The No-Go List (Never Automate):

    Strictly exclude legal, privacy, terms of service, or compliance pages.

    Exclude temporary event pages (e.g., "2024 Conference Agenda").

    Never automate direct content rewrites or programmatic publishing based solely on this score.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatic safeguards for downstream automation pipelines
nogo_keywords = ['privacy', 'terms', 'legal', 'compliance', '2024-event']

def check_automation_safety(url_slug):
    if any(keyword in url_slug.lower() for keyword in nogo_keywords):
        return "BLOCKED: Matches No-Go List"
    return "SAFE_FOR_QUEUE"

print("Safety check test on '/legal-terms':", check_automation_safety("/legal-terms"))

Safety check test on '/legal-terms': BLOCKED: Matches No-Go List


## 4. Monitoring / retrain triggers


    Monitoring Triggers: Track the top 100 queued items. If fewer than 50% result in traffic recoveries 30 days post-refresh, flag the pipeline for a performance review.

    Retrain Triggers: Retrain the model quarterly, or immediately if major SERP layout changes (like a confirmed Google Core Update) are deployed globally.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define trigger thresholds for MLOps monitoring
retrain_triggers = {
    "post_refresh_success_rate_drop": "< 50%",
    "time_elapsed": "> 90 days",
    "external_event": "Google Core Update"
}
print("MLOps Retrain Triggers Defined:", retrain_triggers)

MLOps Retrain Triggers Defined: {'post_refresh_success_rate_drop': '< 50%', 'time_elapsed': '> 90 days', 'external_event': 'Google Core Update'}


## 5. Exports for the paper
Exports: Generating action_playbook_queue.csv containing the top 100 ranked URLs, along with an age distribution figure. These artifacts will be directly referenced in the final research paper.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import os
import matplotlib.pyplot as plt

# Ensure directories exist
os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

# Load data and apply scoring logic
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['baseline_score'] = (df['content_age_days'] * 0.5) + (df['impressions_90d'] * 0.1)

# Sort and select top 100
export_df = df.sort_values(by='baseline_score', ascending=False).head(100).copy()
export_df['action_label'] = 'QUEUE_FOR_EDITORIAL_REFRESH'
export_df['reason_code'] = 'STALE_HIGH_EXPOSURE'

# Export CSV
export_path = "../outputs/action_playbook_queue.csv"
export_df.to_csv(export_path, index=False)
print(f"✅ Exported top 100 queue to {export_path}")

# Generate and save figure
plt.figure(figsize=(8, 4))
plt.hist(export_df['content_age_days'].dropna(), bins=20, color='#2c3e50', edgecolor='white')
plt.title('Age Distribution of Top 100 Queued Pages')
plt.xlabel('Content Age (Days)')
plt.ylabel('Frequency')
fig_path = "../figures/queued_age_distribution.png"
plt.savefig(fig_path)
plt.close()
print(f"✅ Exported figure to {fig_path}")

✅ Exported top 100 queue to ../outputs/action_playbook_queue.csv
✅ Exported figure to ../figures/queued_age_distribution.png


## Self-check

Before you submit, confirm each line honestly:

[x] Every section above is filled — markdown thinking AND the code that backs it

[x] The notebook runs top to bottom with no errors (Runtime → Run all)

[x] No client names, URLs, or private queries anywhere

[x] My claims use careful words: observed, measured, directional, decision-support

[x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.